# 02. 量子化で L4 に載る「いちばん大きいモデル」を試す

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の2つ目の実験です。

使うモデル: [`google/gemma-4-31B-it`](https://huggingface.co/google/gemma-4-31B-it)（約31B = 310億パラメータ、Apache-2.0）

- そのまま（bf16）だと約 62GB。L4（約22GB）には載らない
- **4bit 量子化**（bitsandbytes NF4）で約 18〜19GB に縮めて、L4 1枚に全部載せる
- 見積もりの根拠は [docs/results/02_quantization_max.md](../docs/results/02_quantization_max.md)

「想定どおり」とは次の4つがすべて満たされることです。

1. モデル全体が GPU に載る（CPU に逃がさない）
2. 読み込み後の VRAM 使用量が 20GB 以下
3. 日本語で筋の通った返事が返る
4. 生成が止まらずに終わる

所要時間の目安: ダウンロード 62GB を含めて 15〜30 分。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU・ディスク・メモリを確認する

In [ ]:
import shutil, subprocess, torch

assert torch.cuda.is_available(), "GPUにつながっていません。ランタイムを L4 GPU にしてください。"
gpu_name = torch.cuda.get_device_name(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
disk_free_gb = shutil.disk_usage("/").free / 1024**3
ram_total_gb = int(open("/proc/meminfo").read().split()[1]) / 1024**2

print("GPU      :", gpu_name)
print("VRAM GB  :", round(vram_total_gb, 1))
print("Disk free:", round(disk_free_gb, 1), "GB（ダウンロードに約 62GB 使う）")
print("RAM GB   :", round(ram_total_gb, 1))
assert disk_free_gb > 70, "ディスクが足りません"

## 2. ライブラリを入れる

Gemma 4 には新しい `transformers` が必要です。`bitsandbytes` は 4bit 量子化に使います。

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
import transformers, bitsandbytes
print("transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

## 3. 4bit 量子化して読み込む

- `load_in_4bit=True` + `nf4`: 重みを 4bit にする
- `bnb_4bit_use_double_quant=True`: 量子化の補助データもさらに縮める
- `bnb_4bit_compute_dtype=bfloat16`: 計算は bf16 で行う
- `device_map={"": 0}`: **全部 GPU 0 に載せる**。載らなければここでエラーになる（＝想定外）

ダウンロードに時間がかかります（10〜20分）。

In [ ]:
import time
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-31B-it"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    dtype=torch.bfloat16,
    device_map={"": 0},
)
load_min = (time.time() - t0) / 60

devices = sorted({str(p.device) for p in model.parameters()})
footprint_gb = model.get_memory_footprint() / 1024**3
vram_after_load_gb = torch.cuda.memory_allocated() / 1024**3

print("読み込み完了:", MODEL_ID)
print("読み込み時間 :", round(load_min, 1), "分（ダウンロード込み）")
print("パラメータの置き場:", devices)
print("モデルのサイズ:", round(footprint_gb, 1), "GB")
print("VRAM 使用量 :", round(vram_after_load_gb, 1), "GB /", round(vram_total_gb, 1), "GB")

## 4. 日本語で聞く

01 と同じ質問に加えて、少し考える質問をもう1つ聞きます。

In [ ]:
def ask(prompt, max_new_tokens=400):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer in Japanese."},
        {"role": "user", "content": prompt},
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)
    n_in = inputs["input_ids"].shape[-1]
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize(); sec = time.time() - t0
    n_out = out.shape[-1] - n_in
    raw = processor.decode(out[0][n_in:], skip_special_tokens=False)
    try:
        parsed = processor.parse_response(raw, prefix=inputs["input_ids"])
        text = parsed.get("content", raw) if isinstance(parsed, dict) else str(parsed)
    except Exception:
        text = processor.decode(out[0][n_in:], skip_special_tokens=True)
    return text.strip(), n_out, sec

torch.cuda.reset_peak_memory_stats()
qa = []
for p in [
    "小学校の児童にも分かる言葉で、GPUとVRAMの違いを3文で説明してください。",
    "りんごが3個入った箱が4つあります。そこから5個食べて、2個もらいました。りんごは何個ですか。考え方も短く書いてください。",
]:
    text, n_out, sec = ask(p)
    qa.append((p, text, n_out, sec))
    print("質問:", p)
    print("返事:", text)
    print(f"({n_out} トークン / {sec:.1f} 秒 = {n_out/sec:.1f} トークン/秒)")
    print("-" * 40)

vram_peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print("生成中の VRAM ピーク:", round(vram_peak_gb, 1), "GB")

## 5. 想定どおりか判定して、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta

checks = {
    "モデル全体が GPU に載った": devices == ["cuda:0"],
    "読み込み後の VRAM が 20GB 以下": vram_after_load_gb <= 20,
    "日本語の返事が返った": all(any("\u3040" <= ch <= "\u30ff" for ch in t) for _, t, _, _ in qa),
    "生成が最後まで終わった": all(n > 0 for _, _, n, _ in qa),
}
ok = all(checks.values())

now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
lines = [
    "# 実行記録: 02 量子化で載る最大モデル",
    "",
    f"- 実行日: {now}",
    "- 実行場所: Google Colab",
    f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
    f"- モデル: {MODEL_ID}",
    "- 量子化: bitsandbytes 4bit NF4（double quant, 計算 bf16）",
    f"- transformers {transformers.__version__} / bitsandbytes {bitsandbytes.__version__} / torch {torch.__version__}",
    f"- 読み込み時間: {round(load_min, 1)} 分（ダウンロード込み）",
    f"- モデルのサイズ: {round(footprint_gb, 1)} GB",
    f"- 読み込み後の VRAM: {round(vram_after_load_gb, 1)} GB",
    f"- 生成中の VRAM ピーク: {round(vram_peak_gb, 1)} GB",
    f"- 想定どおりか: {'はい' if ok else 'いいえ'}",
    "",
    "## 判定",
    "",
] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + [""]
for p, t, n, s in qa:
    lines += [f"## 質問: {p}", "", f"速度: {n} トークン / {s:.1f} 秒 = {n/s:.1f} トークン/秒", "", "```", t, "```", ""]
print("\n".join(lines))

## 終わったら

**ランタイム → セッションを管理 → 解放** を必ず押してください。L4 をつないだままだと CU が減ります。